In [1]:
import numpy as np
import pandas as pd

In [2]:
data_ids = [361260, 361259, 361242]
n_ests = [50, 100, 500, 1000]
min_samples_leafs = [1, 5, 10]
max_features = [0.1, 0.33, "1.0"]

In [3]:
# for each data_id, load the result and save as a df
dfs = []
for data_id in data_ids:
    # get number of samples in the data_id by reading X csv
    X = np.loadtxt(f"data/{data_id}/X.csv", delimiter=",")
    n_samples = 1000
    n_features = X.shape[1]
    for n_est in n_ests:
        for min_samples_leaf in min_samples_leafs:
            for max_feature in max_features:
                # create the directory if it doesn't exist
                dir_path = f"results/{data_id}/n_estimators_{n_est}/min_samples_leaf_{min_samples_leaf}/max_features_{max_feature}"
                results_path = f"{dir_path}/runtime_results.csv"
                results_df = pd.read_csv(results_path)
                # divide every col in df except 'data_id' by n_samples
                for col in results_df.columns:
                    if col != 'data_id':
                        results_df[col] = results_df[col] / n_samples
                # add columns for n_estimators, min_samples_leaf, max_features
                results_df['n_estimators'] = n_est
                results_df['min_samples_leaf'] = min_samples_leaf
                results_df['max_features'] = max_feature
                results_df['num_features'] = n_features
                dfs.append(results_df)
df = pd.concat(dfs, ignore_index=True)

In [4]:
df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])

,data_id,rf_fitting_time,rf_plus_baseline_fitting_time,rf_plus_fitting_time,shap_explainer_time,shap_values_time,lime_time,lmdi_baseline_explainer_time,lmdi_baseline_values_time,lmdi_plus_explainer_time,lmdi_plus_values_time,n_estimators,min_samples_leaf,max_features,num_features
82,361242,0.000993,0.004479,0.008863,0.000013,0.002975,0.176910,0.000005,0.016729,0.000008,0.025225,100,1,0.33,81
46,361259,0.000818,0.005737,0.010714,0.000009,0.003598,0.129684,0.000006,0.006415,0.000013,0.007560,100,1,0.33,32
10,361260,0.000302,0.008938,0.010193,0.000010,0.003283,0.092636,0.000006,0.002966,0.000014,0.003208,100,1,0.33,15
85,361242,0.001334,0.009553,0.017130,0.000011,0.003984,0.273451,0.000007,0.027807,0.000010,0.035832,100,5,0.33,81
49,361259,0.000641,0.005609,0.010538,0.000012,0.003305,0.129364,0.000007,0.008894,0.000022,0.010962,100,5,0.33,32
13,361260,0.000240,0.008094,0.007386,0.000005,0.002666,0.082713,0.000006,0.002627,0.000014,0.002818,100,5,0.33,15
88,361242,0.001039,0.005682,0.008988,0.000013,0.002969,0.176426,0.000005,0.016767,0.000008,0.025272,100,10,0.33,81
52,361259,0.000690,0.007153,0.010808,0.000009,0.003612,0.130533,0.000006,0.006621,0.000013,0.007978,100,10,0.33,32
16,361260,0.000304,0.008656,0.010147,0.000012,0.003293,0.092581,0.000006,0.002961,0.000014,0.003218,100,10,0.33,15


In [5]:
display_df = df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, lime_time, shap_values_time, rf_plus_fitting_time + lmdi_plus_values_time
display_df = display_df[['data_id', 'num_features', 'min_samples_leaf', 'lime_time', 'shap_values_time', 'rf_plus_fitting_time', 'lmdi_plus_values_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_values_time'], inplace=True)
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lime_time': 'LIME',
    'shap_values_time': 'TreeSHAP',
    'lmdi_plus_time': 'LMDI+'
})
display_df

,OpenML Data ID,# of Features,Min. Samples per Leaf,LIME,TreeSHAP,LMDI+
82,361242,81,1,0.176910,0.002975,0.034089
46,361259,32,1,0.129684,0.003598,0.018274
10,361260,15,1,0.092636,0.003283,0.013402
85,361242,81,5,0.273451,0.003984,0.052962
49,361259,32,5,0.129364,0.003305,0.021501
13,361260,15,5,0.082713,0.002666,0.010204
88,361242,81,10,0.176426,0.002969,0.034259
52,361259,32,10,0.130533,0.003612,0.018786
16,361260,15,10,0.092581,0.003293,0.013365


In [6]:
# round to fourth decimal place
display_df = display_df.round(4)

In [7]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   Min. Samples per Leaf |   LIME |   TreeSHAP |   LMDI+ |
|-----------------:|----------------:|------------------------:|-------:|-----------:|--------:|
|           361242 |              81 |                       1 | 0.1769 |     0.003  |  0.0341 |
|           361259 |              32 |                       1 | 0.1297 |     0.0036 |  0.0183 |
|           361260 |              15 |                       1 | 0.0926 |     0.0033 |  0.0134 |
|           361242 |              81 |                       5 | 0.2735 |     0.004  |  0.053  |
|           361259 |              32 |                       5 | 0.1294 |     0.0033 |  0.0215 |
|           361260 |              15 |                       5 | 0.0827 |     0.0027 |  0.0102 |
|           361242 |              81 |                      10 | 0.1764 |     0.003  |  0.0343 |
|           361259 |              32 |                      10 | 0.1305 |     0.0036 |  0.0188 |
|           361260 |          

In [8]:
df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5)].sort_values(by=['n_estimators', 'data_id'])

,data_id,rf_fitting_time,rf_plus_baseline_fitting_time,rf_plus_fitting_time,shap_explainer_time,shap_values_time,lime_time,lmdi_baseline_explainer_time,lmdi_baseline_values_time,lmdi_plus_explainer_time,lmdi_plus_values_time,n_estimators,min_samples_leaf,max_features,num_features
76,361242,0.001403,0.007771,0.017648,0.000011,0.004242,0.264910,0.000007,0.022317,0.000009,0.033008,50,5,0.33,81
40,361259,0.000761,0.007495,0.014952,0.000012,0.004020,0.146737,0.000007,0.007329,0.000014,0.008617,50,5,0.33,32
4,361260,0.000284,0.008801,0.010231,0.000012,0.003134,0.094373,0.000007,0.003009,0.000014,0.003215,50,5,0.33,15
85,361242,0.001334,0.009553,0.017130,0.000011,0.003984,0.273451,0.000007,0.027807,0.000010,0.035832,100,5,0.33,81
49,361259,0.000641,0.005609,0.010538,0.000012,0.003305,0.129364,0.000007,0.008894,0.000022,0.010962,100,5,0.33,32
13,361260,0.000240,0.008094,0.007386,0.000005,0.002666,0.082713,0.000006,0.002627,0.000014,0.002818,100,5,0.33,15
94,361242,0.001096,0.007227,0.012661,0.000009,0.003747,0.226547,0.000007,0.021185,0.000008,0.031533,500,5,0.33,81
58,361259,0.000778,0.007520,0.014962,0.000012,0.004024,0.148386,0.000007,0.007210,0.000014,0.008409,500,5,0.33,32
22,361260,0.000248,0.005731,0.006992,0.000010,0.002728,0.078060,0.000006,0.002617,0.000013,0.002810,500,5,0.33,15
103,361242,0.001311,0.008490,0.015364,0.000009,0.004281,0.268799,0.000008,0.028898,0.000010,0.041047,1000,5,0.33,81


In [9]:
display_df = df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5) & (df['n_estimators'] != 50)].sort_values(by=['n_estimators', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, lime_time, shap_values_time, rf_plus_fitting_time + lmdi_plus_values_time
display_df = display_df[['data_id', 'num_features', 'n_estimators', 'lime_time', 'shap_values_time', 'rf_plus_fitting_time', 'lmdi_plus_values_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_values_time'], inplace=True)
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lime_time': 'LIME',
    'shap_values_time': 'TreeSHAP',
    'lmdi_plus_time': 'LMDI+'
})
display_df

,OpenML Data ID,# of Features,# of Estimators,LIME,TreeSHAP,LMDI+
85,361242,81,100,0.273451,0.003984,0.052962
49,361259,32,100,0.129364,0.003305,0.021501
13,361260,15,100,0.082713,0.002666,0.010204
94,361242,81,500,0.226547,0.003747,0.044195
58,361259,32,500,0.148386,0.004024,0.023371
22,361260,15,500,0.078060,0.002728,0.009803
103,361242,81,1000,0.268799,0.004281,0.056411
67,361259,32,1000,0.131595,0.003454,0.018379
31,361260,15,1000,0.078788,0.003020,0.010332


In [10]:
# round to fourth decimal place
display_df = display_df.round(4)

In [11]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   # of Estimators |   LIME |   TreeSHAP |   LMDI+ |
|-----------------:|----------------:|------------------:|-------:|-----------:|--------:|
|           361242 |              81 |               100 | 0.2735 |     0.004  |  0.053  |
|           361259 |              32 |               100 | 0.1294 |     0.0033 |  0.0215 |
|           361260 |              15 |               100 | 0.0827 |     0.0027 |  0.0102 |
|           361242 |              81 |               500 | 0.2265 |     0.0037 |  0.0442 |
|           361259 |              32 |               500 | 0.1484 |     0.004  |  0.0234 |
|           361260 |              15 |               500 | 0.0781 |     0.0027 |  0.0098 |
|           361242 |              81 |              1000 | 0.2688 |     0.0043 |  0.0564 |
|           361259 |              32 |              1000 | 0.1316 |     0.0035 |  0.0184 |
|           361260 |              15 |              1000 | 0.0788 |     0.003  |  0.0103 |